# Assignment 6

In [ ]:
# import libraries
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.optimize import root_scalar


## 1 Physics of Vibrations

In [ ]:
# constants in system
m1, m2 = 1.0, 1.0 #kg, masses
k1, k2, k3 = 10.0, 10.0, 10.0 #N/m, spring constants

# defining the stiffness matrix
K = np.array([
[(k1 + k2)/m1, -k2/m1],
[-k2/m2, (k2 + k3)/m2]
])

# solve for eigenvalues
evals, evecs = np.linalg.eig(K)

# sort eigenvalues and vectors, ascending
idx = np.argsort(evals)
evals = evals[idx]
evecs = evecs[:, idx]

# calculate omega (eigenvalues are omega squared)
omegas = np.sqrt(evals)

# diaplay values of omegas
for i in range(len(omegas)):
    print(f'Mode {i+1}: Frequency = {omegas[i]:.3f} rad/s')
    print(f'Relative Motion [x1, x2] = {evecs[:, i]}')

# times
t = np.linspace(0, 10, 500)
def get_mode_motion(t, omega, eigenvector):
    x1 = eigenvector[0] * np.cos(omega * t)
    x2 = eigenvector[1] * np.cos(omega * t)
    return x1, x2

# plotting displacement
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# MODE 1 (Symmetric / Lower Frequency)
x1_m1, x2_m1 = get_mode_motion(t, omegas[0], evecs[:, 0])
ax1.plot(t, x1_m1, 'b-', label='Mass 1')
ax1.plot(t, x2_m1, 'r--', label='Mass 2')
ax1.set_title(f"Mode 1: Symmetric Motion ($\omega$ = {omegas[0]:.2f} rad/s)")
ax1.set_ylabel("Displacement")
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# MODE 2 (Anti-Symmetric / Higher Frequency)
x1_m2, x2_m2 = get_mode_motion(t, omegas[1], evecs[:, 1])
ax2.plot(t, x1_m2, 'b-', label='Mass 1')
ax2.plot(t, x2_m2, 'r--', label='Mass 2')
ax2.set_title(f"Mode 2: Anti-Symmetric Motion ($\omega$ = {omegas[1]:.2f} rad/s)")
ax2.set_ylabel("Displacement")
ax2.set_xlabel("Time (s)")
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### 1.a.1 What happens if masses are different?
When the masses are different, their maximum displacements are not equal. In first mode, their oscillations are in sync, but the larger mass (m1) displaces less than the smaller mass (m2). Conversely, the larger mass displaces much more than the smaller mass in the second mode.

#### 1.a.2 Predict what you expect, and then explain you see.

I predict the larger mass will move less due to Newton's second law. Because the m1 has greater mass, it experiences less acceleration from a given force compared to m2. So, in the first mode, m1 has less initial acceleration, so its displacement remains smaller than m2. While in the second mode, m1 has far more inital acceleration, so when the system changes direction, it takes more time before force is sufficient to change direction.

#### 1.a.3
Changing the value of the middle spring constant (k2) changes frequency. Increasing the spring constant, increases the frequency and decreasing the spring constant. Very small middle spring constant, decreases the frequency to a limit of about 1.6 cycles per second.

#### 1.a.4
When all spring constants are doubled frequency increases from about 2/3 to 2 cycles per seconds.

In [ ]:
# 
N = 10
m = 1.0
k = 100.0

diag_val = 2*k/m
off_diag = -k/m

# defining stiffness matrix
A = np.diag(np.full(N, diag_val)) + \
    np.diag(np.full(N-1, off_diag), k=1) + \
    np.diag(np.full(N-1, off_diag), k=-1)

evals, evecs = np.linalg.eig(A)

# solve for eigenvalues and vectors
idx = np.argsort(evals)
omegas = np.sqrt(evals[idx])
evecs = evecs[:, idx]

# plotting displacements
fig, axes = plt.subplots(10, 1, figsize=(10, 10), sharex=True)
x_pos = np.arange(1, N + 1)

for i in range(10):
    ax = axes[i]
    # The i-th eigenvector tells us the displacement of each mass
    mode_shape = evecs[:, i]
    
    # Plot the "Envelope" of the vibration
    ax.plot(x_pos, mode_shape, 'o-', markersize=8, label=f"Mode {i+1}")
    ax.axhline(0, color='black', alpha=0.3)
    ax.set_ylabel("Amplitude")
    ax.set_title(f"Mode {i+1}: $\omega$ = {omegas[i]:.2f} rad/s")
    ax.grid(True, alpha=0.2)

    
axes[-1].set_xlabel("Mass Index")
plt.tight_layout()
plt.show()

print(f"Constructed a {N}x{N} matrix.")
print("Mode 1 is the fundamental frequency (like a guitar string).")
print("Higher modes have more 'nodes' (crossings of the zero line).")


#### 1.b.1
The first 4 modes have m-1 nodes, where m is mode number. The tenth mode is expected to have 10-1=9 modes.

#### 1.b.2
In the tenth mode masses tend to displace in the opposite direction from each other.

In [ ]:
print(evecs.shape)
print(evecs[0].shape)
print(evecs[4][2])
print(evecs)

#### 1.c.1
The eigenvector of the first mode is ```evecs[0]```.

#### 1.c.2
The displacement of the third mass in the fifth mode is ```evecs[4][2]```.



## 2 Static Systems

In [ ]:
# create coefficient matrix
I_coeff  =  np.array([
    [10, -2, 0],
    [-2, 8, -4],
    [0, -4, 6]
])

# create constant matrix
I_const = np.array([[12], [0], [-5]])

# solve for and display currents
I_sols = np.linalg.solve(I_coeff, I_const)
for i, I_vals in enumerate(I_sols):
    current = round(I_vals[0], 2)
    print(f'I{i+1} =', current, 'A')


In [ ]:
W = 1 #N, weight
t01_deg = 30 #deg, angle of first rope
t02_deg = 45 #deg, angle of second rope

# converting angles to radians
t01 = np.radians(t01_deg)
t02 = np.radians(t02_deg)

# create coefficient matrix
T_coeff = np.array([
    [-np.cos(t01), np.cos(t02)],
    [np.sin(t01), np.sin(t02)]
    ])

# create constant matrix
T_const = np.array([[0], [W]])

# solve for and display tensions
T_sols = np.linalg.solve(T_coeff, T_const)
for T, T_vals in enumerate(T_sols):
    tension = round(T_vals[0], 2)
    print(f'T{T+1} =', tension, 'N')

## 3 Duffing Oscillator

In [ ]:
# define parameters
alpha = 1
beta = 0.5

# define x as symbol
x = sp.symbols('x')

# create voltage function and take derivate for force
V = (alpha * x**2)/2 + (beta * x**4)/4
dVdx = sp.diff(-V, x)

# convert to numpy function
f_force = sp.lambdify(x, dVdx, 'numpy')

# get force and position 
x_vals = np.linspace(-2, 2, 100)
y_vals = f_force(x_vals)

# plot
plt.plot(x_vals, y_vals)
plt.title('Force versus Position')
plt.grid()


## 4 Damped Oscillators



In [ ]:
# define x as function
x_func = sp.Function('x')

# define parameters as symbols
t, gam, ome = sp.symbols('t gamma omega')

# set up derivatives
X, X_t, X_tt = x_func(t), x_func(t).diff(t), x_func(t).diff(t, t)

# set up differential equations
diffeq = sp.Eq(X_tt + gam*X_t + (ome**2)*X, 0)


In [ ]:
# solve for general solution
gen_solution = sp.dsolve(diffeq)
gen_solution

In [ ]:
# substitute in specific values
spc_solution =spc_solution = gen_solution.subs({gam: 5, ome: 2})
spc_solution

#### 4.d
The solution contains exponentials.

## 5 Quantum Well

In [ ]:
# define variables
E = sp.symbols('E')

# parameter values
h_bar = 6.626e-34/(2*math.pi) #kg*m^2/s, reduced planck constant
L = 10e-9 #m, length of well
v0 = 50 #J?, energy depth of well
me = 9.11e-31 #kg, mass of electron

# create function and substitute values
fE = sp.sqrt(E) * sp.tan(sp.sqrt( 2*E*me*L**2/h_bar**2 )) - sp.sqrt(v0 - E)

# convert expression to function
fE_func = sp.lambdify(E, fE, 'numpy')

# plot root function 
E_vals = np.linspace(0, 10, 101)
fE_vals = fE_func(E_vals)

plt.plot(E_vals, fE_vals)
plt.title('Root Function vs Energies')

In [ ]:
# roots
sol = root_scalar(fE_func, bracket=[1.3, 10], method='bisect')
print(sol.root)